# Microcosmic God → Catch Transfer Test

This notebook tests whether brains evolved in the [microcosmic-god](https://github.com/asystemoffields/microcosmic-god) artificial-life sandbox have transferable representations — does the cognition that emerged in a complex causal-survival world transfer to a simple paddle-and-ball game it has never seen?

## The protocol

1. Load a brain from a microcosmic-god checkpoint. **Brain weights stay frozen — no further learning.**
2. Train a small linear adapter that maps Catch's 4-dim observation into the brain's 72-dim input space, and the brain's 15-dim output into Catch's 3 actions.
3. Compare three conditions:
   - **Random policy** — left/stay/right uniformly. Lower bound.
   - **Untrained brain + trained adapter** — brain weights random-init; only adapter learns. Controls for what the adapter alone can do.
   - **Trained brain (loaded checkpoint) + trained adapter** — the actual transfer test.

If condition 3 outperforms condition 2 meaningfully, the brain's internal representations are doing useful work in a task they were never trained on. That's a real signal of transferability.

**Why a linear adapter and not a deeper one?** Deep adapters can learn to compute Catch from scratch, hiding any (lack of) contribution from the brain. A small linear projection is the cleanest test: the brain has to be doing something useful for the adapter to extract a working policy.

## Setup

Pure numpy + matplotlib. Clones the microcosmic-god repo to access sample checkpoints.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('microcosmic-god'):
    !git clone -q https://github.com/asystemoffields/microcosmic-god.git

BRAIN_REPO = 'microcosmic-god' if os.path.exists('microcosmic-god') else '.'
print('Using brain repo at:', BRAIN_REPO)

## The Catch environment

10×10 grid. Paddle on bottom row, 3 cells wide. Ball drops one row per tick from a random column. Three actions: left / stay / right. Reward +1 if caught, −1 if missed. Each episode is exactly 9 steps (the ball reaches the paddle row).

Observation: `[paddle_x, ball_x, ball_y, t]` all normalized to [0, 1] — 4 floats.

In [ ]:
class Catch:
    def __init__(self, size=10, paddle_width=3, rng=None):
        self.size = size
        self.paddle_width = paddle_width
        self.rng = rng if rng is not None else np.random.default_rng()
        self.reset()

    def reset(self):
        self.paddle_x = self.size // 2
        self.ball_x = int(self.rng.integers(0, self.size))
        self.ball_y = 0
        self.t = 0
        return self._obs()

    def _obs(self):
        return np.array([
            self.paddle_x / max(1, self.size - 1),
            self.ball_x / max(1, self.size - 1),
            self.ball_y / max(1, self.size - 1),
            self.t / max(1, self.size - 1),
        ], dtype=np.float64)

    def step(self, action: int):
        if action == 0:
            self.paddle_x = max(0, self.paddle_x - 1)
        elif action == 2:
            self.paddle_x = min(self.size - 1, self.paddle_x + 1)
        self.ball_y += 1
        self.t += 1
        done = self.ball_y >= self.size - 1
        reward = 0.0
        if done:
            half = self.paddle_width // 2
            distance = abs(self.ball_x - self.paddle_x)
            reward = 1.0 if distance <= half else -1.0
        return self._obs(), reward, done

## Brain inference

Loads a microcosmic-god `TinyBrain` checkpoint. Pure numpy, forward-pass only — no plasticity. Mirrors the math in `microcosmic_god/brain.py`'s `forward()` and `_attend()`. Attention noise is set to zero for deterministic evaluation (we're testing what the brain *learned*, not its noise tolerance).

In [ ]:
class TinyBrainInference:
    def __init__(self, weights_in, weights_out, bias_h, bias_o,
                 attention_weights=None, attention_bias=None):
        self.weights_in = weights_in
        self.weights_out = weights_out
        self.bias_h = bias_h
        self.bias_o = bias_o
        self.attention_weights = attention_weights if (attention_weights is not None and attention_weights.size) else None
        self.attention_bias = attention_bias if (attention_bias is not None and attention_bias.size) else None
        self.input_size = weights_in.shape[1]
        self.hidden_size = weights_in.shape[0]
        self.output_size = weights_out.shape[0]
        self.hidden = np.zeros(self.hidden_size)

    @classmethod
    def from_checkpoint(cls, path):
        with open(path) as f:
            data = json.load(f)
        # Microcosmic-god checkpoints wrap the brain in 'brain' or 'brain_template'.
        brain_data = data.get('brain') or data.get('brain_template') or data
        h = int(brain_data['hidden_size'])
        i = int(brain_data['input_size'])
        o = int(brain_data['output_size'])
        weights_in = np.array(brain_data['weights_in'], dtype=np.float64).reshape(h, i)
        weights_out = np.array(brain_data['weights_out'], dtype=np.float64).reshape(o, h)
        bias_h = np.array(brain_data['bias_h'], dtype=np.float64)
        bias_o = np.array(brain_data['bias_o'], dtype=np.float64)
        aw = brain_data.get('attention_weights') or []
        ab = brain_data.get('attention_bias') or []
        attention_weights = np.array(aw, dtype=np.float64).reshape(h, i) if len(aw) == h * i else None
        attention_bias = np.array(ab, dtype=np.float64) if len(ab) == i else None
        return cls(weights_in, weights_out, bias_h, bias_o, attention_weights, attention_bias)

    @classmethod
    def random(cls, input_size, hidden_size, output_size, rng=None, with_attention=True):
        rng = rng or np.random.default_rng(0)
        weights_in = rng.normal(0, 0.45, (hidden_size, input_size))
        weights_out = rng.normal(0, 0.45, (output_size, hidden_size))
        bias_h = rng.normal(0, 0.08, hidden_size)
        bias_o = rng.normal(0, 0.08, output_size)
        if with_attention:
            attention_weights = rng.normal(0, 0.15, (hidden_size, input_size))
            attention_bias = rng.normal(3.0, 0.10, input_size)
        else:
            attention_weights = None
            attention_bias = None
        return cls(weights_in, weights_out, bias_h, bias_o, attention_weights, attention_bias)

    def reset(self):
        self.hidden = np.zeros(self.hidden_size)

    def _attend(self, x, budget_fraction=0.95):
        if self.attention_weights is None:
            return x
        inv_h = 1.0 / np.sqrt(max(1, self.hidden_size))
        logits = self.attention_bias + (self.hidden @ self.attention_weights) * inv_h
        np.clip(logits, -30.0, 30.0, out=logits)
        raw = 1.0 / (1.0 + np.exp(-logits))
        budget = self.input_size * budget_fraction
        total = raw.sum()
        fidelity = raw * (budget / total) if total > budget and total > 0 else raw
        return x * fidelity  # noise=0 in transfer mode

    def forward(self, inputs):
        x = self._attend(np.asarray(inputs, dtype=np.float64))
        inv = 1.0 / np.sqrt(max(1, self.input_size))
        new_hidden = np.tanh(self.bias_h + 0.62 * self.hidden + (self.weights_in @ x) * inv)
        self.hidden = new_hidden
        inv_h = 1.0 / np.sqrt(max(1, self.hidden_size))
        return self.bias_o + (self.weights_out @ self.hidden) * inv_h

## The adapter

Two trainable linear projections wrapped around the (frozen) brain:
- `W_in` — Catch's 4-dim observation → brain's input space (72 dims by default)
- `W_out` — brain's 15-dim output → 3 Catch actions

The brain itself is unchanged. If the brain's hidden representations are useful for Catch, training only these projections should produce a competent player.

In [ ]:
class CatchAdapter:
    def __init__(self, brain, catch_obs_size=4, catch_action_size=3, rng=None):
        self.brain = brain
        self.catch_obs_size = catch_obs_size
        self.catch_action_size = catch_action_size
        self.rng = rng or np.random.default_rng()
        self.W_in = self.rng.normal(0, 0.30, (brain.input_size, catch_obs_size))
        self.b_in = np.zeros(brain.input_size)
        self.W_out = self.rng.normal(0, 0.30, (catch_action_size, brain.output_size))
        self.b_out = np.zeros(catch_action_size)

    def reset(self):
        self.brain.reset()

    def policy_logits(self, catch_obs):
        brain_in = self.W_in @ catch_obs + self.b_in
        brain_out = self.brain.forward(brain_in)
        return self.W_out @ brain_out + self.b_out

    def act_greedy(self, catch_obs):
        return int(np.argmax(self.policy_logits(catch_obs)))

    def act_stochastic(self, catch_obs, temperature=1.0):
        logits = self.policy_logits(catch_obs) / max(1e-6, temperature)
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        return int(self.rng.choice(len(probs), p=probs))

    def get_params(self):
        return self.W_in.copy(), self.W_out.copy(), self.b_in.copy(), self.b_out.copy()

    def set_params(self, W_in, W_out, b_in, b_out):
        self.W_in = W_in
        self.W_out = W_out
        self.b_in = b_in
        self.b_out = b_out

## Training (Evolution Strategies)

We can't backprop through the brain in pure numpy without a deep-learning library, and we don't want to — the test is purest if the brain is genuinely frozen with no gradient flow. Instead we use Evolution Strategies (ES): perturb the adapter weights with small gaussian noise, evaluate, and update toward perturbations that improved performance. Slow but framework-free, and keeps the brain truly frozen.

In [ ]:
def run_episode(adapter, env, greedy=False):
    adapter.reset()
    obs = env.reset()
    total = 0.0
    done = False
    while not done:
        action = adapter.act_greedy(obs) if greedy else adapter.act_stochastic(obs, temperature=1.0)
        obs, reward, done = env.step(action)
        total += reward
    return total

def evaluate(adapter, env, n_episodes=200):
    return float(np.mean([run_episode(adapter, env, greedy=True) for _ in range(n_episodes)]))

def train_es(adapter, env, generations=150, pop=32, sigma=0.20, lr=0.20,
             n_avg=3, log_every=10, seed=0, verbose=True):
    """Evolution Strategies. n_avg episodes are averaged per perturbation
    to reduce variance from Catch's stochastic ball position - without
    this the gradient estimate is too noisy and learning thrashes."""
    base = adapter.get_params()
    history = []
    rng = np.random.default_rng(seed)
    for gen in range(generations):
        perturbations = []
        rewards = []
        for _ in range(pop):
            eps = tuple(rng.normal(0, sigma, p.shape) for p in base)
            adapter.set_params(*[b + e for b, e in zip(base, eps)])
            r = float(np.mean([run_episode(adapter, env, greedy=False) for _ in range(n_avg)]))
            perturbations.append(eps)
            rewards.append(r)
        rewards = np.array(rewards)
        if rewards.std() > 1e-8:
            normed = (rewards - rewards.mean()) / rewards.std()
        else:
            normed = rewards - rewards.mean()
        deltas = [np.zeros_like(p) for p in base]
        for r, eps in zip(normed, perturbations):
            for i in range(len(deltas)):
                deltas[i] += r * eps[i]
        base = tuple(b + (lr / (pop * sigma)) * d for b, d in zip(base, deltas))
        adapter.set_params(*base)
        if gen % log_every == 0 or gen == generations - 1:
            avg = evaluate(adapter, env, n_episodes=40)
            history.append((gen, float(avg)))
            if verbose:
                print(f'  gen {gen:3d}: eval reward {avg:+.3f}')
    return adapter, history

## The experiment

In [ ]:
# Default sample: a learner-champion brain (hidden_size=14, attention head present)
# from the 30-min seed-1 run. Try 'final_overall_champion.json' or
# 'final_tool_champion.json' to compare different selection-pressure outputs.
SAMPLE_BRAIN = os.path.join(BRAIN_REPO, 'transfer/sample_brains/learner_champion_hidden14.json')

def random_baseline(env, n_episodes=200, seed=0):
    rng = np.random.default_rng(seed)
    rewards = []
    for _ in range(n_episodes):
        env.reset()
        total = 0.0
        done = False
        while not done:
            action = int(rng.integers(0, 3))
            _, r, done = env.step(action)
            total += r
        rewards.append(total)
    return float(np.mean(rewards))

env = Catch(rng=np.random.default_rng(7))

print('=== 1) Random policy ===')
random_score = random_baseline(env, n_episodes=200)
print(f'Random policy: {random_score:+.3f}\n')

print('=== 2) Untrained brain + trained adapter (control) ===')
random_brain = TinyBrainInference.random(input_size=72, hidden_size=14, output_size=15,
                                          rng=np.random.default_rng(7), with_attention=True)
control_adapter = CatchAdapter(random_brain, rng=np.random.default_rng(11))
control_adapter, control_history = train_es(control_adapter, env, generations=150, pop=32, seed=0)
control_score = evaluate(control_adapter, env, n_episodes=200)
print(f'Untrained brain + adapter: {control_score:+.3f}\n')

print('=== 3) Trained brain + trained adapter (transfer test) ===')
print(f'Loading brain from {SAMPLE_BRAIN}')
trained_brain = TinyBrainInference.from_checkpoint(SAMPLE_BRAIN)
print(f'  hidden={trained_brain.hidden_size}, input={trained_brain.input_size}, output={trained_brain.output_size}')
print(f'  attention head present: {trained_brain.attention_weights is not None}\n')
trained_adapter = CatchAdapter(trained_brain, rng=np.random.default_rng(11))
trained_adapter, trained_history = train_es(trained_adapter, env, generations=150, pop=32, seed=0)
trained_score = evaluate(trained_adapter, env, n_episodes=200)
print(f'Trained brain + adapter:   {trained_score:+.3f}')

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if control_history:
    gens, scores = zip(*control_history)
    ax.plot(gens, scores, label='untrained brain + adapter (control)', marker='o')
if trained_history:
    gens, scores = zip(*trained_history)
    ax.plot(gens, scores, label='trained brain + adapter (transfer)', marker='o')
ax.axhline(random_score, color='gray', linestyle='--', label=f'random ({random_score:+.2f})')
ax.set_xlabel('Generation')
ax.set_ylabel('Mean reward (eval)')
ax.set_title('Microcosmic God brain → Catch transfer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nFinal scores (mean reward over 200 episodes, greedy policy):')
print(f'  Random policy:             {random_score:+.3f}')
print(f'  Untrained brain (control): {control_score:+.3f}')
print(f'  Trained brain (transfer):  {trained_score:+.3f}')
print(f'\n  Transfer advantage (trained − control): {trained_score - control_score:+.3f}')

## Interpreting the result

**Reward range:** Catch returns +1 on a catch, −1 on a miss, so per-episode reward is in {−1, +1}. A perfect player scores +1.0; random scores around 0.0 (2/3 chance of being out of reach × −1, 1/3 hit × +1, depending on paddle width and start position).

**What the conditions tell you:**
- If `trained > control`: the brain's representations are doing useful work. **Transfer worked.** The cognition that emerged in microcosmic-god generalizes.
- If `trained ≈ control`: the brain isn't contributing much. The adapter is solving Catch on its own through the brain's random nonlinearity. No real transfer.
- If `control > random` and `trained > random`: the adapter alone can solve some of Catch (linear projection through any nonlinear bottleneck has decent capacity).

**The interesting case is `trained > control`.** This says the brain learned representations that are useful beyond their training environment — exactly what "transferable minds" means in the project's framing.

Try it with different sample brains in `transfer/sample_brains/` — final_overall_champion vs final_tool_champion etc. — to see if different selection pressures produce more or less transferable representations.